<a href="https://colab.research.google.com/github/jihye-jeon2/python/blob/main/%EB%AC%B8%EC%9E%90%EC%97%B4_%EB%B2%94%EC%A3%BC%ED%98%95%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

환경설정


In [15]:
#구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

#드라이브에 저장된 폰트 등록
import matplotlib as mpl
import matplotlib.pyplot as plt  #그래프를그리는 pylpot모듈
import matplotlib.font_manager as fm #폰트를 관리하는 모듈

#드라이브내 폰트경로
font_path='/content/drive/MyDrive/kwu (1)/빅데이터/dataPreProcesing/NanumGothic.ttf'



fm.fontManager.addfont(font_path)
mpl.rc('font',family='NanumGothic') #matplotlib 기본 폰트로 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.sans-serif']=['NanumGothic','sans-serif']
plt.rcParams['axes.unicode_minus'] = False #마이너스기호가 깨지지않도록 유니코드 마이너스 비활성화

print("현재폰트:  ",plt.rcParams['font.family'])  #현재 적용된 폰트 이름 출력


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재폰트:   ['NanumGothic']


In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore') #모든 경고 메시지를 무시하도록 설정

#타이타닉 데이터 로드
titanic=pd.read_csv('/content/drive/MyDrive/kwu (1)/빅데이터/dataPreProcesing/train.csv')
print(titanic.shape)
titanic.head()
titanic.tail()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [21]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder,OrdinalEncoder

#LabelEncoder : 범주형 문자열을 정수로변환, 순서형에서 사용
#OneHotEncoder : 순서가없는명목형 변수에 사용
#OrdinalEncoder : 순서가 잇는범주형변수에 사용

In [22]:
#결측치 처리
titanic['Age']=titanic.groupby(['Sex','Pclass'])['Age'].transform(
  lambda x: x.fillna(x.median())
)
titanic['Embarked']=titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])

#Cabin 결측치처리 -> 열 삭제\
titanic=titanic.drop(columns=['Cabin'])
titanic['Title']=titanic['Name'].str.extract(r',\s*([^\.]+)\.').squeeze() #DataFrame->Series 로 변환
titanic['Title']=titanic['Title'].apply(
    lambda x: x if x in ['Mr','Miss','Mrs','Master'] else 'Other'
)
titanic['AgeGroup']=pd.cut(
    titanic['Age'], bins=[0,12,17,59,100], labels=['아동','청소년','성인','노인'
    ]
)
titanic['FamilySize']=titanic['SibSp']+titanic['Parch']+1 #가족수
print(titanic[['Sex','Embarked','Pclass','AgeGroup','Title']].head())

      Sex Embarked  Pclass AgeGroup Title
0    male        S       3       성인    Mr
1  female        C       1       성인   Mrs
2  female        S       3       성인  Miss
3  female        S       1       성인   Mrs
4    male        S       3       성인    Mr


str 접근자로 문자열 정제

In [27]:
#공백 , 대소문자 처리 -> Male =   MALE로 인식하게
sample=pd.Series(['  Male  ','female', 'MALE' ,'Female  '])
print('strip:',sample.str.strip().tolist()) #공백 지우기 strip()  리스트로 출력하기 tolist()
print('lower:',sample.str.lower().tolist)  #소문자 변환

#한꺼번에 코드
print('strip+lower:',sample.str.strip().str.lower().tolist())

strip: ['Male', 'female', 'MALE', 'Female']
lower: <bound method IndexOpsMixin.tolist of 0      male  
1      female
2        male
3    female  
dtype: object>
strip+lower: ['male', 'female', 'male', 'female']


str.contains() / str.extract() 로 패턴탐색

In [28]:
has_mr = titanic['Name'].str.contains('Mr\.')  #이름에 포함하고 잇는가 /포함여부
has_miss=titanic['Name'].str.contains('Miss\.')
print(f'Mr 호칭: {has_mr.sum()}명')
print(f"Miss 호칭: {has_miss.sum()}명")

Mr 호칭: 517명
Miss 호칭: 182명


In [29]:
titanic['TicketType']=titanic['Ticket'].str.match(r'^\d+$').map(
    {True:'숫자형',False:'문자포함형'}
)  #정규표현식 정수\d로 시작(^)하구 $는 끝나기 첨부터 끝까지 숫자가 잇는 값에 Match되면 True, 경우에 맞게 mapping
print('\n티켓 유형별 생존율:')
print(titanic.groupby('TicketType'['Survived'].mean().round(3)))


레이블 인코딩 -map()/replace()

In [31]:
#인공지능이 그그 인식할수잇도록 문자-> 숫자변환
#map() 메서드
titanic['Sex_encoded']=titanic['Sex'].map({'male':0,'female':1})
print('성별인코딩:')
print(titanic[['Sex','Sex_encoded']].value_counts().reset_index())

#replace로 치환, 원본 값 유지 옵션 잇음
titanic['Embarked_encoded']=titanic['Embarked'].replace({'S':0,'C':1,'Q':2})
titanic[['Embarked','Embarked_encoded']].head()

#숫자가 더 크면 ai가 중요도를 잘못 인식할수잇음 -> 원-핫 인코딩

성별인코딩:
      Sex  Sex_encoded  count
0    male            0    577
1  female            1    314


,Embarked,Embarked_encoded
0,S,0
1,C,1
2,S,0
3,S,0
4,S,0


원-핫 인코딩 -pd.get_dummies() 순서와 상관없는 명목형데이터에 사용됨, 혈액형  같은거 숫자와 상관이 없는 중요도 / 순서가 잇다 -> 1등석 2등석 숫자와 상관이잇는 중요도

In [32]:
embarked_dummies=pd.get_dummies(titanic['Embarked'],prefix='embarked') #prefix=접두사, titanic['Embarked']값 앞에다가 embarked를 접두사로 붙일거다
print('원-핫 인코딩결과:')
print(embarked_dummies.head())

원-핫 인코딩결과:
   embarked_C  embarked_Q  embarked_S
0       False       False        True
1        True       False       False
2       False       False        True
3       False       False        True
4       False       False        True


In [34]:
embarked_dummies_drop=pd.get_dummies(titanic['Embarked'],prefix='embarked',drop_first=True) #첫번째 열을 삭제함 각 행에 어차피 True는 하나라서 두개만 잇어도 추론 가능 -> 데이터차원 줄일수잇음
print('원-핫 인코딩결과:')
print(embarked_dummies_drop.head())

원-핫 인코딩결과:
   embarked_Q  embarked_S
0       False        True
1       False       False
2       False        True
3       False        True
4       False        True


In [35]:
#여러 열 한번에 인코딩 /인공지능이 알아먹을수잇는 데이터 만들기
cols_to_encode=['Sex','Embarked','Title'] #원-핫 인코딩 적용할열들을 리스트 형태로
titanic_ohe=pd.get_dummies(titanic,columns=cols_to_encode,drop_first=False)  #맞다 1, 아니다 0 싹다 변환
print(titanic_ohe.head())

   PassengerId  Survived  Pclass  ... Title_Mr  Title_Mrs  Title_Other
0            1         0       3  ...     True      False        False
1            2         1       1  ...    False       True        False
2            3         1       3  ...    False      False        False
3            4         1       1  ...    False       True        False
4            5         0       3  ...     True      False        False

[5 rows x 24 columns]


원-핫 인코딩-sklearn OneHotEncoder

In [39]:
#sklearn OneHotEncoder : 훈련/테스트 분리 시 유용(새범주 대응가능)
from sklearn.model_selection import train_test_split

X=titanic[['Sex','Embarked']].copy()
X_train, X_test=train_test_split(X, test_size=0.2,random_state=42)   # 0.2는 20%는 테스트용 데이터로 뺀다  80%는 훈련

#fit()은 훈련데이터로만, 훈련 데이터의 범주 목록 학습
ohe=OneHotEncoder(
    sparse_output =False,  #결과를 평범한 숫자표 (numpy array?)로 반환
    drop='first',  #첫번째 카테고리는 삭제하라
    handle_unknown='ignore'  #알려지지않은 새 범주는 ignore한다 / 오류없이 0으로 처리하겟다
)
ohe.fit(X_train)  #train데이터를 학습시킴 , 훈련

# transform() 훈련데이터 학습된 범주 기준으로 원-핫 인코딩 / 0,1 등 인공지능이 알수잇는 숫자로 변환하는거다  #transform()
X_train_ohe=ohe.transform(X_train)
X_test_ohe=ohe.transform(X_test)

print('생성된 열:', ohe.get_feature_names_out()) #새로 만들어진 열 이름이 무엇인가
print('훈련 데이터 shape:',X_train_ohe.shape) #데이터 모양 shape출력하라
print('\n 결과예시(첫 3행):')
result_df=pd.DataFrame(X_train_ohe,columns=ohe.get_feature_names_out()) #DataFrame으로 만들고 columns이름표 붙여주기
print(result_df.head(3))

생성된 열: ['Sex_male' 'Embarked_Q' 'Embarked_S']
훈련 데이터 shape: (712, 3)

 결과예시(첫 3행):
   Sex_male  Embarked_Q  Embarked_S
0       1.0         0.0         1.0
1       1.0         0.0         1.0
2       1.0         0.0         1.0


순서형 인코딩-직접매핑

In [41]:
#순서형(Ordinal) 인코딩: 범주 간 의미 잇는 순서가 잇을때 순서를 반영해서 점수부여
#순서가 잇다 -> 숫자가 높으면 머신러닝이 중요한데이터로 인식함  1등석 , 2등석 -> 숫자가 높으면중요한 애니까 매핑을 직접해줌
titanic['Pclass_encoded']=titanic['Pclass'].map({1:2,2:1,3:0})
print('객실 등급순서형 인코딩:')
print(titanic[['Pclass','Pclass_encoded']].drop_duplicates().sort_values('Pclass'))   #drop_duplicates()중복제거 메서드. Pclass를 기준으로 정렬하라

#나이그룹을 순서형 인코딩 직접매핑
age_order={'아동':0,'청소년':1,'성인':2,'노인':3}
titanic['AgeGroup_encoded']=titanic['AgeGroup'].map(age_order)
print('\n나이구간순서형인코딩:')
print(titanic[['AgeGroup','AgeGroup_encoded']].drop_duplicates().sort_values('AgeGroup_encoded').to_string())


객실 등급순서형 인코딩:
   Pclass  Pclass_encoded
1       1               2
9       2               1
0       3               0

나이구간순서형인코딩:
   AgeGroup AgeGroup_encoded
7        아동                0
9       청소년                1
0        성인                2
33       노인                3


순서형 인코딩- sklearn OrdinalEncoder 을 사용한다 (순서가 중요한 애들)

In [43]:
#OrdinalEncoder순서형 배열은 2차원 입력 필요하므로 [[]]사용
oe=OrdinalEncoder(
    categories=[['아동','청소년','성인','노인']],  #0,1,2,3
    handle_unknown='use_encoded_value' , #학습 시 없엇던 새 범주가 등장하면 오류대신 unknown_value로 처리
    unknown_value=-1  #새로운 범주에 부여할값
)

age_group_col=titanic[['AgeGroup']].astype(str) #나이그룹데이터를 str문자열로
titanic['AgeGroup_ordinal']=oe.fit_transform(age_group_col) #transform 정해준 숫자로 바꿔서 새 열에 저장
print(titanic[['AgeGroup','AgeGroup_ordinal']].drop_duplicates().sort_values('AgeGroup_ordinal').to_string())   #중복제거, 우리가 매긴 순서대로 정렬해서출력하라



   AgeGroup  AgeGroup_ordinal
7        아동               0.0
9       청소년               1.0
0        성인               2.0
33       노인               3.0


빈도 인코딩 & 타깃 인코딩(고급)

In [ ]:
#빈도인코딩: 각 범주의 등장횟수(빈도)로 대체  얼마나 흔한 그룹인가
freq_map=titanic['Title'].value_counts().to_dict()  #호칭 별로 몇명이나 잇는지 세어서 사전형태로
titanic['Title_freq']=titanic['Title'].map(freq_map)  #호칭을 인원수 숫자로 바꿔라 map() 메서드
print("빈도 인코딩 결과:")
print(titanic[['Title','Title_freq']].drop_duplicates().sort_values('Title_freq',ascending=False)) #내림차순으로, 인원수 많은 순대로

#타깃인코딩: 각 범주의 목표변수(생존율) 평균으로 대체
target_map=titanic.groupby('Title')['Survived'].mean().to_dict()  #해당 호칭의 생존율을 호칭별 평균 생존율로
titanic['Title_target']=titanic['Title'].map(target_map) #Title글자를 평균생존율로 바꾸기 (숫자로)
print("\n타깃 인코딩 결과(생존율 기반):")
print(titanic[['Title','Title_target']].drop_duplicates().sort_values('Title_target',ascending=False).round(3))

타이타닉 전처리

In [46]:
titanic_final=pd.read_csv('/content/drive/MyDrive/kwu (1)/빅데이터/dataPreProcesing/train.csv')
titanic_final=titanic_final.drop(columns=['Cabin'])
#결측치처리
titanic_final['Age']=titanic_final.groupby(['Sex','Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
titanic_final['Embarked']=titanic_final['Embarked'].fillna(
    titanic_final['Embarked'].mode()[0]
)
#이상치처리 ->캡핑
Q1,Q3 = titanic_final['Fare'].quantile([0.25,0.75])
titanic_final['Fare']=titanic_final['Fare'].clip(upper=Q3 + 1.5 * (Q3-Q1)) #초과 값을 상한으로 대체
#스케일링 분포변환 -로그변환
titanic_final['Fare']=np.log1p(titanic_final['Fare'])  #양의왜도

#파생변수 생성
titanic_final['FamilySize']=titanic_final['SibSp']+titanic_final['Parch']+1
titanic_final['IsAlone']=(titanic_final['FamilySize']==1).astype(int)
titanic_final['Title']=titanic_final['Name'].str.extract(r',\s*([^\].+)\').squeeze()
titanic_final['Title']=titanic_final['Title'].apply(
    lambda x: x if x in ['Mr','Miss','Mrs','Master'] else 'Other'
)
#범주형 인코딩
#명목형 변수(순서없음 )->원-핫 인코딩
titanic_final=pd.get_dummies(titanic_final,columns=['Sex','Embarked','Title'],drop_first=True)
#순서형 변수(순서잇음)->순서형인코딩
titanic_final['Pclass']=titanic_final['Pclass'].map({1:2,2:1,3:0})

#불필요 열 제거
titanic_final=titanic_final.drop(columns=['Name','Ticket','SibSp','Parch'])
#스케일링
from sklearn.preprocessing import StandardScaler #평균=0,표준편차=1로 표준화
scaler=StandardScaler()
titanic_final[num_cols]=scaler.transform(titanic_final[nums_cols])

#최종결과 확인
print(f"\n최종 shape:{titanic_final.shape}")
print(f'결측치 : {titanic_final.isnull().sum().sum()}개')
print(f'\n최종칼럼목록:')
print(titanic_final.columns.tolist())
print(f'\n 데이터 미리보기')
print(titanic_final.head(3).round(3))

SyntaxError: unterminated string literal (detected at line 19) (1635647774.py, line 19)